# Claude Code + AgentCore Gateway (Native OAuth) — Open Insurance

This notebook configures **Claude Code** to connect to the AgentCore Gateway using its **native MCP OAuth flow**. When Claude Code connects to the Gateway, it:

1. Receives a `401` with OAuth metadata pointing to the Okta authorization server
2. Opens a **browser tab** for Okta login (Authorization Code + PKCE)
3. Receives the JWT and caches it for subsequent requests
4. Gateway enforces **Cedar policies** based on the authenticated user's identity

No token management code needed — Claude Code handles the entire OAuth lifecycle.

```
Claude Code
  │
  │ MCP StreamableHTTP → 401 → OAuth discovery
  │
  │ Browser popup → Okta login (Authorization Code + PKCE)
  │
  │ JWT Bearer token (cached automatically)
  ▼
AgentCore Gateway (CUSTOM_JWT authorizer)
  ├── Validates JWT (signature, audience, client_id)
  ├── Cedar Policy Engine (ENFORCE mode)
  │     principal: AgentCore::OAuthUser::"<JWT sub>"
  │
  ├───────────────┐
  ▼               ▼
Lambda          Lambda
(PolicyLookup)  (ClaimsData)
```

## Prerequisites

- **`00_okta_setup.ipynb`** completed (auth server, SPA app, default scopes)
- **`01_agentcore_setup.ipynb`** completed (Gateway, Lambda targets, Cedar policies)
- **`gateway_config.json`** exists with deployed resource IDs
- **Claude Code** installed (`npm install -g @anthropic-ai/claude-code`)
- **`OKTA_SPA_CLIENT_ID`** in `.env` (created by `00_okta_setup.ipynb` Cell 4)

## Cell 1: Load Configuration

Load the Gateway config from `01_agentcore_setup.ipynb` and the SPA client ID from `.env`.

In [ ]:
import os
import json
import requests
import boto3
from dotenv import load_dotenv

load_dotenv(override=True)

# --- Load Gateway config from 01_agentcore_setup ---
with open("gateway_config.json") as f:
    config = json.load(f)

GATEWAY_ID = config["gateway_id"]
GATEWAY_URL = config["gateway_url"]
OKTA_DOMAIN = config["okta_domain"]
OKTA_ISSUER = config["okta_issuer"]
AWS_REGION = config["aws_region"]

# --- SPA client ID (created by 00_okta_setup Cell 4) ---
SPA_CLIENT_ID = os.environ["OKTA_SPA_CLIENT_ID"]
CALLBACK_PORT = 8400

print(f"Gateway ID:     {GATEWAY_ID}")
print(f"Gateway URL:    {GATEWAY_URL}")
print(f"Okta Domain:    {OKTA_DOMAIN}")
print(f"Okta Issuer:    {OKTA_ISSUER}")
print(f"AWS Region:     {AWS_REGION}")
print(f"SPA Client ID:  {SPA_CLIENT_ID}")
print(f"\nOAuth Discovery (Gateway -> Okta):")

# Verify the Gateway returns OAuth metadata
resp = requests.get(f"{GATEWAY_URL.replace('/mcp', '')}/.well-known/oauth-protected-resource", timeout=10)
print(f"  Protected Resource Metadata: {resp.json()}")

## Cell 2: Update Gateway `allowedClients`

The AgentCore Gateway validates the JWT `client_id` claim against its `allowedClients` list. We add the SPA client ID so tokens issued by the Claude Code app are accepted.

In [4]:
import time

agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

# --- Get current Gateway config ---
gw = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
current_clients = gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedClients"]
print(f"Current allowedClients: {current_clients}")

# --- Add SPA client ID if not already present ---
if SPA_CLIENT_ID in current_clients:
    print(f"\nSPA client ID already in allowedClients — no update needed")
else:
    new_clients = current_clients + [SPA_CLIENT_ID]
    response = agentcore.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=gw["name"],
        roleArn=gw["roleArn"],
        protocolType=gw["protocolType"],
        authorizerType=gw["authorizerType"],
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw["authorizerConfiguration"]["customJWTAuthorizer"]["discoveryUrl"],
                "allowedAudience": gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedAudience"],
                "allowedClients": new_clients,
            }
        },
        policyEngineConfiguration=gw.get("policyEngineConfiguration", {}),
        exceptionLevel="DEBUG",
    )
    print(f"Updated allowedClients: {new_clients}")
    print(f"Gateway status: {response['status']}")

    # Wait for READY
    for attempt in range(12):
        gw = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
        if gw["status"] == "READY":
            print("Gateway READY")
            break
        time.sleep(5)
        print(f"  Status: {gw['status']} ({(attempt + 1) * 5}s)")

Current allowedClients: ['0oa116s4xvkFwMbSw698']
Updated allowedClients: ['0oa116s4xvkFwMbSw698', '0oa126ujb31nR6Jrc698']
Gateway status: UPDATING
  Status: UPDATING (5s)
Gateway READY


## Cell 3: Configure Claude Code MCP Server

This adds the AgentCore Gateway as an MCP server in Claude Code's project config.

**Key config fields:**
- `type: http` — MCP StreamableHTTP transport
- `url` — Gateway's MCP endpoint
- `oauth.clientId` — Okta SPA app client ID (public client, no secret)
- `oauth.callbackPort` — Port for the OAuth redirect callback
- `oauth.scope` — OAuth scopes to request (`openid` for OIDC, `groups` for Cedar identity)
- `oauth.authorizationUrl` / `oauth.tokenUrl` — Explicit Okta endpoints (required because Claude Code's MCP OAuth discovery sends a `resource` parameter that Okta doesn't handle without explicit scopes)

In [ ]:
import subprocess

mcp_config = {
    "type": "http",
    "url": GATEWAY_URL,
    "oauth": {
        "clientId": SPA_CLIENT_ID,
        "callbackPort": CALLBACK_PORT,
        "scope": "openid groups",
        "authorizationUrl": f"{OKTA_ISSUER}/v1/authorize",
        "tokenUrl": f"{OKTA_ISSUER}/v1/token",
    },
}

print("MCP Server config:")
print(json.dumps(mcp_config, indent=2))

# --- Add to Claude Code (project scope) ---
result = subprocess.run(
    ["claude", "mcp", "add-json", "agentcore-gateway", json.dumps(mcp_config)],
    capture_output=True, text=True,
)

if result.returncode == 0:
    print(f"\nAdded MCP server to Claude Code")
    print(result.stdout.strip())
else:
    # May already exist — try removing and re-adding
    subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)
    result = subprocess.run(
        ["claude", "mcp", "add-json", "agentcore-gateway", json.dumps(mcp_config)],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f"\nUpdated MCP server in Claude Code")
        print(result.stdout.strip())
    else:
        print(f"\nError: {result.stderr}")

# --- WORKAROUND: Claude Code may not persist the 'scope' field ---
# Patch .claude.json directly to ensure scope and URLs are present
claude_json_path = os.path.expanduser("~/.claude.json")
print(f"\nClaude Json Path: {claude_json_path}")
with open(claude_json_path) as f:
    claude_config = json.load(f)

project_key = os.getcwd()
if "projects" in claude_config and project_key in claude_config["projects"]:
    servers = claude_config["projects"][project_key].get("mcpServers", {})
    if "agentcore-gateway" in servers:
        servers["agentcore-gateway"]["oauth"]["scope"] = "openid groups"
        servers["agentcore-gateway"]["oauth"]["authorizationUrl"] = f"{OKTA_ISSUER}/v1/authorize"
        servers["agentcore-gateway"]["oauth"]["tokenUrl"] = f"{OKTA_ISSUER}/v1/token"
        with open(claude_json_path, "w") as f:
            json.dump(claude_config, f, indent=2)
        print("\nPatched .claude.json with scope and OAuth URLs")

print(f"\n--- Summary ---")
print(f"Gateway:        {GATEWAY_URL}")
print(f"Okta SPA Client: {SPA_CLIENT_ID}")
print(f"Callback Port:  {CALLBACK_PORT}")
print(f"Scopes:         openid groups")

MCP Server config:
{
  "type": "http",
  "url": "https://agentcore-mcp-demo-gateway-ouusdduhmd.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp",
  "oauth": {
    "clientId": "0oa126ujb31nR6Jrc698",
    "callbackPort": 8400,
    "scope": "openid groups",
    "authorizationUrl": "https://trial-8088255.okta.com/oauth2/default/v1/authorize",
    "tokenUrl": "https://trial-8088255.okta.com/oauth2/default/v1/token"
  }
}

Updated MCP server in Claude Code
Added http MCP server agentcore-gateway to local config

--- Summary ---
Gateway:        https://agentcore-mcp-demo-gateway-ouusdduhmd.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
Okta SPA Client: 0oa126ujb31nR6Jrc698
Callback Port:  8400
Scopes:         openid groups


## Cell 4: Save Config

Save the SPA client ID to `gateway_config.json` for reference.

In [ ]:
config["okta_spa_client_id"] = SPA_CLIENT_ID
config["claude_code_callback_port"] = CALLBACK_PORT

with open("gateway_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved SPA client ID to gateway_config.json")
print(f"\n  okta_spa_client_id: {SPA_CLIENT_ID}")
print(f"  claude_code_callback_port: {CALLBACK_PORT}")

## Test It

Open a **new Claude Code session** in this project directory:

```bash
cd /path/to/demo-centralized-mcp-server
claude
```

On startup, Claude Code will connect to the `agentcore-gateway` MCP server:

1. A **browser tab opens** with the Okta login page
2. Log in as your Okta user (e.g., **Alice**, **Bob**, or your contextual demo user)
3. After authentication, Claude Code displays: *"Authentication successful. Connected to agentcore-gateway."*
4. Run `/mcp` to see available tools

### Expected Results by User

| Login as | Tools visible | ClaimsData access |
|----------|--------------|-------------------|
| **Alice** (or any `analysts` user) | `PolicyLookup___lookup_policy` | **BLOCKED** (no Cedar permit) |
| **Bob** (or any `finance-admins` user) | `PolicyLookup___lookup_policy` + `ClaimsData___query_claims` | Full access, no constraints |
| **Contextual user** (`CONTEXTUAL_USER` from `.env`) | `PolicyLookup___lookup_policy` + `ClaimsData___query_claims` | Only when `max_amount <= CONTEXTUAL_MAX_AMOUNT` (Cedar contextual) |

### Demo: Role-Based Access (Alice vs Bob)

```
> Look up insurance policy POL-10042
> Show me the claims summary with adjuster notes
```

As Alice, the claims query fails (Cedar DENY). As Bob, both queries succeed with all 5 claims.

### Demo: Cedar Contextual Control (your contextual demo user)

Log in as your contextual demo user (the Okta user set in `CONTEXTUAL_USER`) and try:

```
> Look up policy POL-10042
```
✅ **Allowed** — PolicyLookup has no restrictions.

```
> Query claims summary with max_amount 50000
```
✅ **Allowed** — Cedar permits because 50000 <= the configured limit. Returns 1 claim (Sarah Chen, $12,400).

```
> Query claims summary with max_amount 200000
```
❌ **Denied by Cedar** — 200000 exceeds the `CONTEXTUAL_MAX_AMOUNT` threshold.

```
> Show me all claims
```
❌ **Denied by Cedar** — `max_amount` field is missing, `when` clause fails.

**The punchline:** Same tool, same Gateway — Cedar dynamically controls *what values* you can pass, not just whether you can call the tool.

### Switching Users

To test as a different user, clear the cached OAuth token and restart:

```bash
# Remove cached OAuth tokens
rm -rf ~/.claude/oauth-tokens/
# Restart Claude Code
claude
```

## Cleanup

Revert the Gateway's `allowedClients` and remove the MCP server from Claude Code.
This does **not** delete the SPA app from Okta (managed by `00_okta_setup.ipynb`).

In [ ]:
# --- Cleanup: revert Gateway and remove MCP server ---

import json
import os
import boto3
import subprocess
from dotenv import load_dotenv

load_dotenv(override=True)

with open("gateway_config.json") as f:
    cfg = json.load(f)

SPA_CLIENT_ID = os.environ.get("OKTA_SPA_CLIENT_ID", cfg.get("okta_spa_client_id", ""))

# 1. Revert Gateway allowedClients (remove SPA client ID)
agentcore = boto3.client("bedrock-agentcore-control", region_name=cfg["aws_region"])
gw = agentcore.get_gateway(gatewayIdentifier=cfg["gateway_id"])
current_clients = gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedClients"]

if SPA_CLIENT_ID and SPA_CLIENT_ID in current_clients:
    new_clients = [c for c in current_clients if c != SPA_CLIENT_ID]
    agentcore.update_gateway(
        gatewayIdentifier=cfg["gateway_id"],
        name=gw["name"],
        roleArn=gw["roleArn"],
        protocolType=gw["protocolType"],
        authorizerType=gw["authorizerType"],
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw["authorizerConfiguration"]["customJWTAuthorizer"]["discoveryUrl"],
                "allowedAudience": gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedAudience"],
                "allowedClients": new_clients,
            }
        },
        policyEngineConfiguration=gw.get("policyEngineConfiguration", {}),
        exceptionLevel="DEBUG",
    )
    print(f"Reverted Gateway allowedClients to: {new_clients}")

# 2. Remove from Claude Code
subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)
print("Removed agentcore-gateway from Claude Code")

# 3. Clean up gateway_config.json
cfg.pop("okta_spa_client_id", None)
cfg.pop("claude_code_callback_port", None)
with open("gateway_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("\nCleanup complete")